# TPC run-by-run QA: rDCA, charge balance, timing and conditions

Read small arrays exported directly from production QA, with at most 100 segments
per run. Current-CDB kEff/GEM-current snapshots and a PHGarfield P/T baseline are
separate from reconstruction provenance: they are **not verified applied values**.


Direct production QA input, at most 100 segments per run. This notebook uses the
working slice-fit threshold of **1000** requested after the pilot. Passing that
threshold is not proof of a good model: every summary retains the actual fit
quality and flags input slices whose deviance/ndf exceeds 10.

For each run, side and pT window we fit
`odd(phi) = c + a sin(phi) + b cos(phi)` with Huber downweighting.
The primary observable is **c = (mu_positive - mu_negative)/2 in cm**.
Tables also give the full difference in cm and mm, avoiding factor-of-two or
unit ambiguity. This is a QA observable; no kEff correction is applied.

Both per-run valid-bin estimates and estimates on a common phi mask are saved.
Reference differences use the common mask. Phi-bin bootstrap errors describe
sensitivity to angular sampling; they are not event/segment bootstrap errors
and do not cover the systematic uncertainty from the slice model.


In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LogNorm

plt.rcParams.update(
    {"figure.figsize": (11, 5), "font.size": 11, "axes.grid": True, "grid.alpha": 0.2}
)

DATA_DIR = Path("input/new/qa-direct-all")
GROUP = "run3pp/ana585_2026p006_v001"
FOCUS_RUN = 81558
FOCUS_WINDOW = "pt_0p5_1"
FOCUS_SIDE = "south"
FOCUS_AXIS = "phi"
FIT_SETTINGS = dict(
    min_counts=100, max_deviance_ndf=1000.0, half_width=None, min_sigma_bins=0.15, max_nfev=350
)
# Set half_width to a positive value in cm to inspect fit-window sensitivity.
PILOT_APPROVED = True
RUN_BATCH = True
SAVE_FIGURES = False
OUTPUT_DIR = Path("rdca_odd_all_runs")
BOOTSTRAP_REPEATS = 100
REFERENCE_RUNS = {"run3pp": 81558, "run3oo": 82688}
# Explicit timing convention: raw sample minus presample/unpacker offset.
# Inspect full spectra before interpreting t0; adjust these windows if needed.
RAW_TIME_SHIFT_BINS = 40.0
TIME_BIN_NS = 56.8
DRIFT_DISTANCE_CM = 102.325  # Fallback; PHGarfield geometry overrides it when available.
T0_WINDOWS = ((2.5, 5.5), (12.0, 16.0), (5.0, 14.0), 8.0)
CM_WINDOWS = ((180.0, 225.0), (300.0, 360.0), (210.0, 290.0), 250.0)

## Load the small dataset and check its provenance
The exporter reads and sums at most 100 original QA histogram files per run.
Every input filename, segment ID, size and modification time is recorded in inputs.json.
No old batch, cache or combined ROOT file is needed. The exporter preserves grouping, original axes, counts, variances, DCA
underflow/overflow and source segment counts. ROOT x/pT/DCA order is converted
explicitly. pT limits must coincide with producer bin edges.

In [ ]:
manifest_path = DATA_DIR / "manifest.json"
if not manifest_path.is_file() or not (DATA_DIR / "SUCCESS").is_file():
    raise FileNotFoundError(f"Copy the completed exported folder here: {DATA_DIR.resolve()}")
MANIFEST = json.loads(manifest_path.read_text())
if (
    MANIFEST.get("schema") != 1
    or MANIFEST.get("fit_performed") is not False
    or MANIFEST.get("input_mode") != "production_qa_histograms"
):
    raise ValueError("Expected direct production-QA export, schema 1 unfitted arrays")
RECORDS = {(r["group"], int(r["run"])): r for r in MANIFEST["runs"]}
COVERAGE = pd.DataFrame(
    [
        dict(
            group=r["group"],
            run=r["run"],
            segments=r["source_segments"],
            maps=len(r["maps"]),
            dedx_status=r["dedx_status"],
            KiB=(DATA_DIR / r["file"]).stat().st_size / 1024,
            issues="; ".join(r["issues"]),
        )
        for r in MANIFEST["runs"]
    ]
)
display(COVERAGE)
print(f"Total portable data: {COVERAGE.KiB.sum() / 1024:.2f} MiB")
print("Selection: historical production QA; no additional dE/dx cut.")
print("Available groups:", sorted({g for g, _ in RECORDS}))
if (GROUP, FOCUS_RUN) not in RECORDS:
    raise ValueError("Select a GROUP/FOCUS_RUN from the coverage table")

_MAP_CACHE = {}
_FIT_CACHE = {}


def get_map(group, run, window, side, axis, charge):
    cache_key = (group, run, window, side, axis, charge)
    if cache_key not in _MAP_CACHE:
        record = RECORDS[(group, run)]
        found = [
            m
            for m in record["maps"]
            if (m["window"], m["side"], m["axis"], m["charge"]) == (window, side, axis, charge)
        ]
        if len(found) != 1:
            raise KeyError(f"Missing/ambiguous map: {cache_key}")
        info = found[0]
        with np.load(DATA_DIR / record["file"], allow_pickle=False) as source:
            data = {
                name: source[info["key"] + "_" + name].copy()
                for name in ("counts", "variance", "xedges", "dcaedges")
            }
        data["info"] = info
        _MAP_CACHE[cache_key] = data
    return _MAP_CACHE[cache_key]


windows = sorted({m["window"] for m in RECORDS[(GROUP, FOCUS_RUN)]["maps"]})
print("Available pT windows:", windows)

## Fit implementation (self-contained)
The model integrates a Gaussian over each DCA bin and adds a constant background
density. It minimizes Poisson deviance including empty bins. Input with variances
inconsistent with count histograms is rejected explicitly. Mean errors use expected
Poisson Fisher information. Every attempted fit retains its candidate parameters
and a reason; a failed fit is never silently replaced by a zero or a full-distribution mean.

The fit window follows the modal core. Change it interactively and inspect individual
slices. A good-looking 2D map alone does not establish that Gaussian fits are usable.

In [ ]:
from scipy.optimize import least_squares
from scipy.special import ndtr, xlogy


def model_counts(theta, edges):
    log_area, mean, log_sigma, background = theta
    sigma = np.exp(log_sigma)
    return np.exp(log_area) * np.diff(ndtr((edges - mean) / sigma)) + background * np.diff(
        edges
    ) / np.median(np.diff(edges))


def fit_slice(
    counts,
    edges,
    variance=None,
    min_counts=100,
    max_deviance_ndf=10.0,
    half_width=None,
    min_sigma_bins=0.15,
    max_nfev=350,
):
    counts = np.asarray(counts, float)
    edges = np.asarray(edges, float)
    result = dict(
        accepted=False,
        reason="unattempted",
        mean_cm=np.nan,
        mean_error_cm=np.nan,
        sigma_cm=np.nan,
        sigma_error_cm=np.nan,
        counts=float(np.sum(counts)),
        deviance_ndf=np.nan,
        fit_low_cm=np.nan,
        fit_high_cm=np.nan,
        theta=None,
        solver_success=False,
        full_mean_cm=np.nan,
        median_cm=np.nan,
    )
    if (
        len(edges) != len(counts) + 1
        or np.any(np.diff(edges) <= 0)
        or np.any(~np.isfinite(counts))
        or np.any(counts < 0)
    ):
        result["reason"] = "invalid_input"
        return result
    centers = (edges[:-1] + edges[1:]) / 2
    total = counts.sum()
    if total > 0:
        result["full_mean_cm"] = float(np.dot(counts, centers) / total)
        result["median_cm"] = float(np.interp(total / 2, np.cumsum(counts), centers))
    if total < min_counts:
        result["reason"] = "low_counts"
        return result
    width = float(np.median(np.diff(edges)))
    peak = int(np.argmax(counts))
    peak_mean = centers[peak]
    baseline = float(np.quantile(counts, 0.25))
    half_height = baseline + 0.5 * (counts[peak] - baseline)
    left = right = peak
    while left > 0 and counts[left] > half_height:
        left -= 1
    while right < len(counts) - 1 and counts[right] > half_height:
        right += 1
    seed_sigma = max(width, (centers[right] - centers[left]) / 2.355)
    half = float(half_width) if half_width is not None else max(4 * width, 3 * seed_sigma)
    if not np.isfinite(half) or half <= 0:
        raise ValueError("half_width must be positive")
    lo = max(0, int(np.searchsorted(edges, peak_mean - half, side="right")) - 1)
    hi = min(len(counts), int(np.searchsorted(edges, peak_mean + half, side="left")))
    n = counts[lo:hi]
    e = edges[lo : hi + 1]
    result.update(fit_low_cm=float(e[0]), fit_high_cm=float(e[-1]))
    if len(n) < 6:
        result["reason"] = "too_few_fit_bins"
        return result
    if n.sum() < min_counts:
        result["reason"] = "low_core_counts"
        return result
    if variance is not None:
        v = np.asarray(variance, float)
        if v.shape != counts.shape or not np.allclose(v[lo:hi], n, rtol=1e-4, atol=1e-5):
            result["reason"] = "weighted_input_not_poisson"
            return result
    sigma_min = min_sigma_bins * width
    sigma_max = float(e[-1] - e[0])
    lower = np.array([np.log(1e-5), e[0], np.log(sigma_min), 0.0])
    upper = np.array(
        [np.log(max(100 * n.sum(), 1e3)), e[-1], np.log(sigma_max), max(10 * max(n), 1.0)]
    )
    initial = np.array(
        [
            np.log(max(n.sum() - baseline * len(n), 1.0)),
            peak_mean,
            np.log(min(seed_sigma, 0.8 * sigma_max)),
            baseline,
        ]
    )
    initial = np.clip(initial, lower + 1e-8, upper - 1e-8)

    def residual(theta):
        expected = np.maximum(model_counts(theta, e), 1e-12)
        deviance = 2 * (expected - n + xlogy(n, n / expected))
        return np.sign(n - expected) * np.sqrt(np.maximum(deviance, 0.0))

    try:
        fit = least_squares(
            residual,
            initial,
            bounds=(lower, upper),
            x_scale="jac",
            max_nfev=max_nfev,
            ftol=1e-8,
            xtol=1e-8,
            gtol=1e-8,
        )
        theta = fit.x
        expected = np.maximum(model_counts(theta, e), 1e-12)
        ndf = len(n) - 4
        result.update(
            theta=theta,
            solver_success=bool(fit.success),
            mean_cm=float(theta[1]),
            sigma_cm=float(np.exp(theta[2])),
            deviance_ndf=float(np.dot(residual(theta), residual(theta)) / ndf),
        )
        # Expected Poisson Fisher information, including an unconstrained local
        # derivative for a background whose optimum can be exactly zero.
        derivatives = []
        for j in range(4):
            step = 1e-5 * max(1.0, abs(theta[j]))
            plus, minus = theta.copy(), theta.copy()
            plus[j] += step
            minus[j] -= step
            derivatives.append((model_counts(plus, e) - model_counts(minus, e)) / (2 * step))
        jacobian = np.column_stack(derivatives)
        scaled = jacobian / np.sqrt(expected[:, None])
        if np.linalg.matrix_rank(scaled) < 4:
            result["reason"] = "singular_covariance"
            return result
        covariance = np.linalg.inv(scaled.T @ scaled)
        result["mean_error_cm"] = float(np.sqrt(max(0.0, covariance[1, 1])))
        result["sigma_error_cm"] = float(np.exp(theta[2]) * np.sqrt(max(0.0, covariance[2, 2])))
        if not fit.success:
            result["reason"] = "solver_failed"
        elif result["sigma_cm"] <= sigma_min * 1.05:
            result["reason"] = "unresolved_width_at_bound"
        elif min(theta[1] - e[0], e[-1] - theta[1]) <= 0.05 * width:
            result["reason"] = "mean_at_fit_boundary"
        elif not (np.isfinite(result["mean_error_cm"]) and result["mean_error_cm"] > 0):
            result["reason"] = "invalid_mean_error"
        elif result["deviance_ndf"] > max_deviance_ndf:
            result["reason"] = "poor_fit_quality"
        else:
            result.update(accepted=True, reason="accepted")
    except (ValueError, FloatingPointError, np.linalg.LinAlgError) as error:
        result["reason"] = "numerical_failure:" + type(error).__name__
    return result


def fit_map(counts, edges, variance=None, **settings):
    return [
        fit_slice(n, edges, None if variance is None else variance[i], **settings)
        for i, n in enumerate(counts)
    ]


def paired_components(positive, negative):
    if len(positive) != len(negative):
        raise ValueError("Charge maps have different coordinate bins")
    valid = np.array(
        [p["accepted"] and n["accepted"] for p, n in zip(positive, negative, strict=True)]
    )
    p = np.array([r["mean_cm"] for r in positive])
    n = np.array([r["mean_cm"] for r in negative])
    error = 0.5 * np.hypot(
        [r["mean_error_cm"] for r in positive], [r["mean_error_cm"] for r in negative]
    )
    even, odd = 0.5 * (p + n), 0.5 * (p - n)
    even[~valid] = odd[~valid] = error[~valid] = np.nan
    return dict(valid=valid, even=even, odd=odd, error=error)


def location_fit(phi, odd, error, min_points=8):
    good = np.isfinite(phi) & np.isfinite(odd) & np.isfinite(error) & (error > 0)
    if good.sum() < min_points:
        return None
    x, y, e = phi[good], odd[good], error[good]
    design = np.column_stack([np.ones(len(x)), np.sin(x), np.cos(x)])
    weights = np.ones(len(x))
    for _ in range(25):
        weighted = design * np.sqrt(weights)[:, None] / e[:, None]
        if np.linalg.matrix_rank(weighted) < 3 or np.linalg.cond(weighted) > 1e8:
            return None
        beta = np.linalg.lstsq(weighted, y * np.sqrt(weights) / e, rcond=None)[0]
        residual = (y - design @ beta) / e
        scale = max(1.0, 1.4826 * np.median(np.abs(residual - np.median(residual))))
        updated = np.minimum(1.0, 2.5 * scale / np.maximum(np.abs(residual), 1e-12))
        if np.max(np.abs(updated - weights)) < 1e-5:
            weights = updated
            break
        weights = updated
    weighted = design * np.sqrt(weights)[:, None] / e[:, None]
    beta = np.linalg.lstsq(weighted, y * np.sqrt(weights) / e, rcond=None)[0]
    residual = (y - design @ beta) / e
    reduced = float(np.sum(weights * residual**2) / (len(x) - 3))
    covariance = np.linalg.inv(weighted.T @ weighted) * max(1.0, reduced)
    return dict(
        offset=float(beta[0]),
        sin1=float(beta[1]),
        cos1=float(beta[2]),
        approximate_error=float(np.sqrt(max(0.0, covariance[0, 0]))),
        npoints=len(x),
        downweighted=int(np.count_nonzero(weights < 0.99)),
        reduced_chi2=reduced,
    )


def summarize_profile(phi, odd, error, valid, bootstrap=100, seed=12345):
    phi, odd, error, valid = map(np.asarray, (phi, odd, error, valid))
    mask = valid & np.isfinite(odd) & np.isfinite(error) & (error > 0)
    result = location_fit(phi[mask], odd[mask], error[mask])
    if result is None:
        return dict(status="insufficient_or_degenerate_phi_coverage", npoints=int(mask.sum()))
    result["status"] = "estimated"
    boot = []
    rng = np.random.default_rng(seed)
    x, y, e = phi[mask], odd[mask], error[mask]
    for _ in range(bootstrap):
        index = rng.integers(0, len(x), len(x))
        fitted = location_fit(x[index], y[index], e[index])
        if fitted is not None:
            boot.append(fitted["offset"])
    enough = len(boot) >= max(20, int(0.8 * bootstrap))
    result["bootstrap_success"] = len(boot)
    result["phi_bootstrap_error"] = float(np.std(boot, ddof=1)) if enough else np.nan
    result["phi_bootstrap_low"] = float(np.quantile(boot, 0.16)) if enough else np.nan
    result["phi_bootstrap_high"] = float(np.quantile(boot, 0.84)) if enough else np.nan
    result["odd_half_cm"] = result["offset"]
    result["odd_half_mm"] = 10 * result["offset"]
    result["full_difference_cm"] = 2 * result["offset"]
    result["full_difference_mm"] = 20 * result["offset"]
    return result


def midpoint_edge(x, counts, left, right, search, expected, direction):
    """50% transition between sideband medians, after a three-bin median filter.

    Width is 10–90% transition width, not a statistical error. Require all three
    crossings in the requested direction and physical order; no nearest-bin fallback.
    """
    x, raw = np.asarray(x, float), np.asarray(counts, float)
    invalid = dict(status="no_resolved_edge", midpoint=np.nan, width_10_90=np.nan)
    if len(x) < 5 or len(x) != len(raw) or not np.isfinite(raw).all():
        return invalid
    y = raw.copy()
    y[1:-1] = np.median(np.vstack([raw[:-2], raw[1:-1], raw[2:]]), axis=0)
    masks = [(x >= a) & (x <= b) for a, b in (left, right)]
    if not all(mask.sum() >= 2 for mask in masks):
        return invalid
    lo, hi = [float(np.median(y[mask])) for mask in masks]
    height = hi - lo
    if direction * height <= 0 or abs(height) < 0.01 * max(abs(lo), abs(hi), 1.0):
        return invalid
    crossings = []
    for fraction in (0.1, 0.5, 0.9):
        target = lo + fraction * height
        good = (x[:-1] >= search[0]) & (x[1:] <= search[1]) & (direction * np.diff(y) > 0)
        good &= (y[:-1] - target) * (y[1:] - target) <= 0
        found = x[:-1][good] + (target - y[:-1][good]) * np.diff(x)[good] / np.diff(y)[good]
        if not len(found):
            return invalid
        crossings.append(float(found[np.argmin(abs(found - expected))]))
    if np.any(np.diff(crossings) < 0):
        return invalid
    return dict(
        status="estimated",
        midpoint=crossings[1],
        width_10_90=abs(crossings[2] - crossings[0]),
        x10=crossings[0],
        x90=crossings[2],
        left_level=lo,
        right_level=hi,
    )


def drift_metrics(
    edges,
    counts,
    source,
    raw_shift=40.0,
    time_bin_ns=56.8,
    distance_cm=102.325,
    t0_windows=((2.5, 5.5), (12.0, 16.0), (5.0, 14.0), 8.0),
    cm_windows=((180.0, 225.0), (300.0, 360.0), (210.0, 290.0), 250.0),
):
    if not (time_bin_ns > 0 and distance_cm > 0 and np.isfinite(raw_shift)):
        raise ValueError("Invalid timing coordinate settings")
    shift = raw_shift if source == "production_thresholded_raw_sample_counts" else 0.0
    x = (np.asarray(edges[:-1]) + np.asarray(edges[1:])) / 2 - shift
    t0 = midpoint_edge(x, counts, *t0_windows, 1)
    cm = midpoint_edge(x, counts, *cm_windows, -1)
    valid = t0["status"] == cm["status"] == "estimated" and cm["midpoint"] > t0["midpoint"]
    drift = (cm["midpoint"] - t0["midpoint"]) * time_bin_ns / 1000 if valid else np.nan
    return dict(
        status="estimated" if valid else "timing_edges_unavailable",
        t0_bins=t0["midpoint"],
        cm_bins=cm["midpoint"],
        t0_width_bins=t0["width_10_90"],
        cm_width_bins=cm["width_10_90"],
        drift_time_us=drift,
        velocity_cm_us=distance_cm / drift if valid else np.nan,
        raw_shift_bins=shift,
        distance_cm=distance_cm,
        time_bin_ns=time_bin_ns,
    )

In [ ]:
def fit_pair(group, run, window, side, axis):
    key = (group, run, window, side, axis, json.dumps(FIT_SETTINGS, sort_keys=True))
    if key not in _FIT_CACHE:
        started = time.monotonic()
        maps = {q: get_map(group, run, window, side, axis, q) for q in ("pos", "neg")}
        p, n = maps["pos"], maps["neg"]
        if not (
            np.array_equal(p["xedges"], n["xedges"])
            and np.array_equal(p["dcaedges"], n["dcaedges"])
        ):
            raise ValueError("Charge-map binning differs")
        fits = {
            q: fit_map(m["counts"], m["dcaedges"], m["variance"], **FIT_SETTINGS)
            for q, m in maps.items()
        }
        components = paired_components(fits["pos"], fits["neg"])
        rows = []
        for charge, results in fits.items():
            for index, result in enumerate(results):
                rows.append(
                    dict(
                        group=group,
                        run=run,
                        window=window,
                        side=side,
                        axis=axis,
                        charge=charge,
                        index=index,
                        **{k: v for k, v in result.items() if k != "theta"},
                    )
                )
        _FIT_CACHE[key] = dict(
            maps=maps,
            fits=fits,
            components=components,
            table=pd.DataFrame(rows),
            elapsed=time.monotonic() - started,
        )
    return _FIT_CACHE[key]


PILOT = fit_pair(GROUP, FOCUS_RUN, FOCUS_WINDOW, FOCUS_SIDE, FOCUS_AXIS)
PILOT_TABLE = PILOT["table"]
display(PILOT_TABLE.groupby(["charge", "reason"]).size().rename("slices").to_frame())
print(
    f"Fit time: {PILOT['elapsed']:.2f}s; usable charge pairs: "
    f"{PILOT['components']['valid'].sum()}/{len(PILOT['components']['valid'])}"
)
display(
    PILOT_TABLE[
        [
            "charge",
            "index",
            "counts",
            "accepted",
            "reason",
            "mean_cm",
            "mean_error_cm",
            "sigma_cm",
            "deviance_ndf",
        ]
    ].head(12)
)

## First inspect maps with accepted and rejected candidate means
Green points are accepted Gaussian core means. Orange crosses are rejected fit
candidates, shown only to diagnose why a profile is empty. The background color is
log(counts). The full-distribution mean is a separate diagnostic and includes tails.

In [ ]:
def plot_pilot_maps(pilot):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, charge in zip(axes, ("pos", "neg")):
        m = pilot["maps"][charge]
        fits = pilot["fits"][charge]
        z = m["counts"].T
        maximum = float(z.max())
        image = ax.pcolormesh(
            m["xedges"],
            m["dcaedges"],
            np.ma.masked_less_equal(z, 0),
            norm=LogNorm(vmin=1, vmax=max(2.0, maximum)),
            cmap="viridis",
            shading="flat",
        )
        x = (m["xedges"][:-1] + m["xedges"][1:]) / 2
        accepted = np.array([r["accepted"] for r in fits])
        mu = np.array([r["mean_cm"] for r in fits])
        err = np.array([r["mean_error_cm"] for r in fits])
        ax.errorbar(
            x[accepted],
            mu[accepted],
            yerr=err[accepted],
            fmt="o",
            ms=3,
            color="lime",
            label=f"Accepted ({accepted.sum()})",
        )
        rejected = (~accepted) & np.isfinite(mu)
        ax.plot(x[rejected], mu[rejected], "x", color="orange", label="Rejected candidate")
        ax.set(xlabel=FOCUS_AXIS, ylabel="DCAxy to origin [cm]", ylim=(-2, 2), title=charge)
        ax.legend(fontsize=9)
        fig.colorbar(image, ax=ax, label="counts")
    fig.suptitle(f"{GROUP}, run {FOCUS_RUN}, {FOCUS_SIDE}, {FOCUS_WINDOW} — existing QA selection")
    fig.tight_layout()
    plt.show()
    plt.close(fig)


plot_pilot_maps(PILOT)

## Inspect individual 1D fits before accepting the pilot
Choose coordinate-bin indices below. Both charges are shown. Dashed candidates
are displayed even when rejected, with the explicit reason and fit quality.
This distinguishes empty inputs, unresolved widths, bad shapes and optimizer failures.

In [ ]:
NCOORD = len(PILOT["maps"]["pos"]["xedges"]) - 1
SLICE_INDICES = sorted(set([0, NCOORD // 4, NCOORD // 2, 3 * NCOORD // 4]))


def plot_slices(pilot, indices):
    fig, axes = plt.subplots(2, len(indices), figsize=(4 * len(indices), 7), squeeze=False)
    for row, charge in enumerate(("pos", "neg")):
        m = pilot["maps"][charge]
        edges = m["dcaedges"]
        centers = (edges[:-1] + edges[1:]) / 2
        for col, index in enumerate(indices):
            if not 0 <= index < len(m["counts"]):
                raise IndexError(index)
            ax = axes[row, col]
            r = pilot["fits"][charge][index]
            n = m["counts"][index]
            ax.errorbar(
                centers, n, yerr=np.sqrt(m["variance"][index]), fmt=".", ms=3, color="black"
            )
            if r["theta"] is not None:
                model = model_counts(r["theta"], edges)
                inside = (centers >= r["fit_low_cm"]) & (centers <= r["fit_high_cm"])
                ax.plot(
                    centers[inside],
                    model[inside],
                    "-" if r["accepted"] else "--",
                    color="tab:green" if r["accepted"] else "tab:orange",
                )
                middle = r["mean_cm"]
                ax.set_xlim(max(edges[0], middle - 1.5), min(edges[-1], middle + 1.5))
                ax.axvspan(r["fit_low_cm"], r["fit_high_cm"], color="tab:blue", alpha=0.06)
            ax.set(
                title=f"{charge}, bin {index}: {r['reason']}\n"
                f"mu={r['mean_cm']:.4f} cm; D/ndf={r['deviance_ndf']:.2f}",
                xlabel="DCAxy [cm]",
                ylabel="counts",
            )
    fig.tight_layout()
    plt.show()
    plt.close(fig)


plot_slices(PILOT, SLICE_INDICES)

In [ ]:
m = PILOT["maps"]["pos"]
x = (m["xedges"][:-1] + m["xedges"][1:]) / 2
components = PILOT["components"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for charge in ("pos", "neg"):
    results = PILOT["fits"][charge]
    good = np.array([r["accepted"] for r in results])
    mu = np.array([r["mean_cm"] for r in results])
    err = np.array([r["mean_error_cm"] for r in results])
    axes[0].errorbar(x[good], mu[good], yerr=err[good], fmt="o-", ms=3, label=charge)
for ax, name in zip(axes[1:], ("even", "odd")):
    ax.errorbar(x, components[name], yerr=components["error"], fmt="o-", ms=3)
    ax.axhline(0, color="gray", ls="--")
    ax.set_title("(mu+ " + ("+" if name == "even" else "-") + " mu-)/2")
axes[0].set_title("Accepted core means")
axes[0].legend()
for ax in axes:
    ax.set(xlabel=FOCUS_AXIS, ylabel="rDCA [cm]")
fig.tight_layout()
plt.show()
plt.close(fig)
print("Odd convention: (positive - negative)/2, in cm. Full difference in mm = 20 * odd_cm.")
if components["valid"].sum() < 8:
    print("STOP before bulk fitting: fewer than eight accepted charge-paired bins in this pilot.")

## Run-wise extraction
Batch fitting is enabled for the requested all-run pass. Fits are cached in memory
by run, side, pT and fit settings. The robust constant uses accepted charge-paired
phi bins only. Runs with fewer than eight usable pairs remain unavailable.

The common mask is the intersection of valid phi bins among eligible runs within
one production, side and pT window. If fewer than eight common bins remain, its
estimate and reference differences remain unavailable; there is no silent fallback.


## Quality and uncertainty
The 1000 threshold is a working QA selection. High-deviance input slices are
counted in the output and the estimates are flagged. Inspect mean stability under
fit-window/model changes before treating the numbers as a calibration.

The uncertainty from the harmonic fit is enlarged for excess angular scatter.
A separate phi-bin bootstrap describes robustness to the sampled angular bins.
Neither substitutes for systematic model checks or segment-level validation.


In [ ]:
import hashlib
from collections import defaultdict

PROFILE_ROWS = []
profile_groups = defaultdict(list)
if not (PILOT_APPROVED and RUN_BATCH):
    print("Batch fitting disabled in configuration.")
else:
    started = time.monotonic()
    for (group, run), record in sorted(RECORDS.items()):
        windows = sorted({m["window"] for m in record["maps"]})
        for window in windows:
            for side in ("south", "north"):
                try:
                    result = fit_pair(group, run, window, side, "phi")
                except KeyError as error:
                    print("Unavailable:", error)
                    continue
                PROFILE_ROWS.append(result["table"])
                m = result["maps"]["pos"]
                phi = (m["xedges"][:-1] + m["xedges"][1:]) / 2
                profile_groups[(group, window, side)].append((run, phi, result))
        print(
            f"{group}/{run}: slice fits complete; elapsed {time.monotonic() - started:.1f}s",
            flush=True,
        )

    scalar_keys = [
        "odd_half_cm",
        "odd_half_mm",
        "full_difference_cm",
        "full_difference_mm",
        "approximate_error",
        "phi_bootstrap_error",
        "phi_bootstrap_low",
        "phi_bootstrap_high",
        "sin1",
        "cos1",
        "reduced_chi2",
        "downweighted",
        "npoints",
        "bootstrap_success",
    ]

    def flat(fit, prefix=""):
        return {
            prefix + "status": fit["status"],
            **{prefix + k: fit.get(k, np.nan) for k in scalar_keys},
        }

    summary = []
    for (group, window, side), profiles in sorted(profile_groups.items()):
        first_phi = profiles[0][1]
        if any(not np.array_equal(phi, first_phi) for _, phi, _ in profiles):
            raise ValueError(f"Incompatible phi axes in {group}/{window}/{side}")
        eligible = [
            result["components"]["valid"]
            for _, _, result in profiles
            if result["components"]["valid"].sum() >= 8
        ]
        common = np.logical_and.reduce(eligible) if eligible else np.zeros(len(first_phi), bool)
        for run, phi, result in profiles:
            c = result["components"]
            seed = int.from_bytes(
                hashlib.sha256(f"{group}/{run}/{window}/{side}".encode()).digest()[:4], "little"
            )
            own = summarize_profile(phi, c["odd"], c["error"], c["valid"], BOOTSTRAP_REPEATS, seed)
            shared = summarize_profile(
                phi, c["odd"], c["error"], c["valid"] & common, BOOTSTRAP_REPEATS, seed
            )
            quality = result["table"]["deviance_ndf"].to_numpy(float)
            finite = quality[np.isfinite(quality)]
            nbad = int(np.count_nonzero(finite > 10.0))
            summary.append(
                dict(
                    group=group,
                    run=run,
                    window=window,
                    side=side,
                    source_segments=RECORDS[(group, run)]["source_segments"],
                    valid_pairs=int(c["valid"].sum()),
                    common_phi_bins=int(common.sum()),
                    fit_threshold=FIT_SETTINGS["max_deviance_ndf"],
                    median_slice_deviance_ndf=float(np.median(finite)) if len(finite) else np.nan,
                    max_slice_deviance_ndf=float(np.max(finite)) if len(finite) else np.nan,
                    slices_above_deviance10=nbad,
                    model_mismatch_flag=bool(nbad),
                    **flat(own),
                    **flat(shared, "common_"),
                )
            )
    PROFILE_SUMMARY = pd.concat(PROFILE_ROWS, ignore_index=True) if PROFILE_ROWS else pd.DataFrame()
    RUN_SUMMARY = pd.DataFrame(summary)
    if not RUN_SUMMARY.empty:
        for column in [
            "reference_run",
            "delta_common_odd_half_mm",
            "delta_common_full_difference_mm",
            "delta_common_approximate_error_mm",
        ]:
            RUN_SUMMARY[column] = np.nan
        for (group, window, side), rows in RUN_SUMMARY.groupby(["group", "window", "side"]):
            reference = REFERENCE_RUNS.get(group.split("/")[0])
            ref = rows[(rows.run == reference) & (rows.common_status == "estimated")]
            if len(ref) != 1:
                print("No usable common-mask reference:", group, window, side, reference)
                continue
            ref = ref.iloc[0]
            good = rows.common_status == "estimated"
            index = rows.index[good]
            RUN_SUMMARY.loc[index, "reference_run"] = reference
            delta = 10 * (rows.loc[index, "common_odd_half_cm"] - ref.common_odd_half_cm)
            error = 10 * np.hypot(
                rows.loc[index, "common_approximate_error"], ref.common_approximate_error
            )
            error.loc[rows.loc[index, "run"] == reference] = 0.0
            RUN_SUMMARY.loc[index, "delta_common_odd_half_mm"] = delta
            RUN_SUMMARY.loc[index, "delta_common_full_difference_mm"] = 2 * delta
            RUN_SUMMARY.loc[index, "delta_common_approximate_error_mm"] = error
        print("Half-difference: (+ minus -)/2. Full difference: + minus -.")
        display(
            RUN_SUMMARY[
                [
                    "group",
                    "run",
                    "side",
                    "window",
                    "valid_pairs",
                    "common_phi_bins",
                    "odd_half_mm",
                    "full_difference_mm",
                    "common_odd_half_mm",
                    "delta_common_odd_half_mm",
                    "model_mismatch_flag",
                ]
            ]
        )
        display(
            RUN_SUMMARY.groupby(["group", "status"])
            .size()
            .rename("run-side-window estimates")
            .to_frame()
        )
        print("Bootstrap errors are phi-bin sensitivity, not segment-level precision.")

## Charge balance from the same small export
Counts below integrate the producer's normal phi/eta bins (eta within [-1.2,1.2]),
whereas the rDCA-versus-phi map has no new eta cut. Keep these selection differences
explicit. Ratios use sums of counts, not means of bin-wise ratios.
Errors use recorded count variances with independent-charge/window propagation;
event-level correlations are not available in these arrays.

In [ ]:
charge_rows = []
for (group, run), record in sorted(RECORDS.items()):
    values = {(r["side"], r["window"], r["charge"]): r for r in record["charge_counts"]}
    for side, window in sorted({(s, w) for s, w, _ in values}):
        p = values.get((side, window, "pos"))
        n = values.get((side, window, "neg"))
        if not p or not n or min(p["count"], n["count"]) <= 0:
            continue
        var_log = p["variance"] / p["count"] ** 2 + n["variance"] / n["count"] ** 2
        ratio = p["count"] / n["count"]
        charge_rows.append(
            dict(
                group=group,
                run=run,
                side=side,
                window=window,
                Npos=p["count"],
                Nneg=n["count"],
                ratio=ratio,
                ratio_error=ratio * np.sqrt(var_log),
                asymmetry=(p["count"] - n["count"]) / (p["count"] + n["count"]),
                asymmetry_error=2
                * np.sqrt(n["count"] ** 2 * p["variance"] + p["count"] ** 2 * n["variance"])
                / (p["count"] + n["count"]) ** 2,
                log_ratio=np.log(ratio),
                log_ratio_error=np.sqrt(var_log),
            )
        )
CHARGE_SUMMARY = pd.DataFrame(charge_rows)
display(CHARGE_SUMMARY)
if not CHARGE_SUMMARY.empty:
    for group, rows in CHARGE_SUMMARY.groupby("group"):
        wide = rows.pivot(index=["run", "side"], columns="window", values="log_ratio")
        if {"pt_0p5_1", "pt_2_4"}.issubset(wide.columns):
            balance = wide["pt_2_4"] - wide["pt_0p5_1"]
            print(group, ": ln(N+/N-)[2,4] - ln(N+/N-)[0.5,1]")
            display(balance.rename("lnr_sl").to_frame())
# Charge balance at every pT window, separately for physical TPC sides.
for group, data in CHARGE_SUMMARY.groupby("group") if not CHARGE_SUMMARY.empty else []:
    runs = sorted(data.run.unique())
    position = {run: i for i, run in enumerate(runs)}
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for ax, side in zip(axes, ("south", "north")):
        for window, rows in data[data.side == side].groupby("window"):
            rows = rows.sort_values("run")
            ax.errorbar(
                [position[r] for r in rows.run],
                rows.ratio,
                yerr=rows.ratio_error,
                fmt="o-",
                ms=3,
                label=window,
            )
        ax.axhline(1, color="gray", ls="--")
        ax.legend()
        ax.set(title=side, ylabel="N+ / N-", xlabel="run")
        ax.set_xticks(range(len(runs)))
        ax.set_xticklabels(runs, rotation=90)
    fig.suptitle(group + " charge balance; stored QA acceptance")
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## Current-CDB conditions and PHGarfield baseline
These are freshly queried snapshots under the explicitly selected global tag.
GEM currents summarize the complete conditions payload, not just the selected QA
segments. The PHGarfield reference uses full-run-average P/T and equal-FEE drift
integration with CDB drift corrections disabled. Its t0 is a configured reference,
not an independently predicted t0 or proof of what production used.

For O+O without a CDB coefficient payload, `keff_reference` uses the explicitly
enabled PHGarfield default GEM-current recipe. `keff_cdb` stays empty and
`keff_kind` labels the estimate as derived. Present invalid CDB coefficients
do not trigger fallback. This does not recover a manual production coefficient
file or per-segment multiplier. The CSV records the recipe constants.


In [ ]:
CONDITIONS = pd.DataFrame()
PHGARFIELD = pd.DataFrame()
condition_info = MANIFEST.get("conditions", {})
if condition_info.get("status") == "exported":
    CONDITIONS = pd.read_csv(DATA_DIR / condition_info["file"])
    if CONDITIONS.duplicated(["group", "run", "side"]).any():
        raise ValueError("Duplicate CDB snapshot rows")
    CONDITIONS["run"] = CONDITIONS["run"].astype(int)
    required = {"keff_cdb", "keff_from_igem", "keff_reference", "keff_kind"}
    if not required.issubset(CONDITIONS.columns):
        raise ValueError(
            "Export fresh conditions with the updated --keff-fallback igem pipeline first."
        )
    if CONDITIONS.keff_applied.notna().any():
        raise ValueError("This snapshot must not claim historical applied kEff")
    display(
        CONDITIONS[
            [
                "group",
                "run",
                "side",
                "cdb_tag",
                "keff_cdb",
                "keff_reference",
                "keff_kind",
                "gem_current_uA",
                "conditions_status",
                "current_scope",
            ]
        ]
    )
    if not RUN_SUMMARY.empty:
        RUN_SUMMARY = RUN_SUMMARY.drop(
            columns=[
                c
                for c in CONDITIONS.columns
                if c not in ["group", "run", "side"] and c in RUN_SUMMARY
            ],
            errors="ignore",
        )
        RUN_SUMMARY = RUN_SUMMARY.merge(
            CONDITIONS, on=["group", "run", "side"], how="left", validate="many_to_one"
        )
    name = condition_info.get("phgarfield_file")
    if name:
        PHGARFIELD = pd.read_csv(DATA_DIR / name)
        if PHGARFIELD.duplicated(["group", "run", "side"]).any():
            raise ValueError("Duplicate PHGarfield reference rows")
        PHGARFIELD["run"] = PHGARFIELD["run"].astype(int)
    for group, data in CONDITIONS.groupby("group"):
        runs = sorted(data.run.unique())
        position = {r: i for i, r in enumerate(runs)}
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
        for side, rows in data.groupby("side"):
            rows = rows.sort_values("run")
            x = [position[r] for r in rows.run]
            axes[0].plot(x, rows.keff_reference, "-", lw=1, label=side)
            direct = rows.keff_kind == "cdb_base_not_verified_applied"
            derived = rows.keff_kind == "derived_igem_not_verified_applied"
            x = np.asarray(x)
            axes[0].plot(x[direct], rows.loc[direct, "keff_reference"], "o", ms=4)
            axes[0].plot(
                x[derived],
                rows.loc[derived, "keff_reference"],
                "^",
                ms=5,
                label=side + " Igem-derived",
            )
            axes[1].errorbar(
                x, rows.gem_current_uA, yerr=rows.gem_current_rms_uA, fmt="o-", ms=3, label=side
            )
        for ax in axes:
            ax.set_xticks(range(len(runs)))
            ax.set_xticklabels(runs, rotation=90)
            ax.legend()
        axes[0].set_ylabel("reference kEff (CDB or labeled Igem recipe)")
        axes[1].set_ylabel("full-payload GEM current [uA]; bars = RMS")
        fig.suptitle(group + "; not verified reconstruction inputs")
        for ax in axes:
            ax.grid(True, alpha=0.2)
        fig.tight_layout()
        plt.show()
        plt.close(fig)
    if not RUN_SUMMARY.empty:
        for (group, window), data in RUN_SUMMARY.groupby(["group", "window"]):
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            for side, rows in data[data.status == "estimated"].groupby("side"):
                for ax, field in zip(axes, ("keff_reference", "gem_current_uA")):
                    ax.scatter(rows[field], rows.odd_half_mm, s=20, label=side)
                    ax.set(
                        xlabel=field + " (current snapshot / recipe)",
                        ylabel="robust rDCA half-odd [mm]",
                    )
            for ax in axes:
                ax.legend()
            fig.suptitle(group + " " + window + "; descriptive comparison, no kEff calibration")
            fig.tight_layout()
            plt.show()
            plt.close(fig)
else:
    print("No current-CDB snapshot requested in this dataset. Rerun the exporter with --cdb-tag.")

## Drift velocity and t0
Production raw-hit timing uses North sectors 0–11 and South sectors 12–23.
Native cleaned timing takes precedence when present. The full spectra are kept;
raw-to-unpacked shift and edge windows are explicit settings at the top.
Missing timing histograms or unresolved edges produce unavailable values.
Inspect the pilot's full spectrum and edge zooms before running every fit.
Velocity is pad-to-CM distance divided by `(CM midpoint - t0 midpoint) * clock`.
PHGarfield geometry supplies that distance when available; otherwise the configured
fallback distance is recorded. The 10–90% edge width is not an uncertainty.


In [ ]:
timing_rows = []
for (group, run), record in sorted(RECORDS.items()):
    for info in record.get(
        "timing", [{"side": s, "status": "not_exported"} for s in ("south", "north")]
    ):
        row = dict(group=group, run=run, side=info["side"], input_status=info["status"])
        reference = pd.DataFrame()
        if not PHGARFIELD.empty:
            reference = PHGARFIELD[
                (PHGARFIELD.group == group)
                & (PHGARFIELD.run == run)
                & (PHGARFIELD.side == info["side"])
            ]
        distance = DRIFT_DISTANCE_CM
        if len(reference) == 1:
            ref = reference.iloc[0]
            distance = float(ref.z_pad_cm - ref.cm_halfwidth_cm)
            # Exporter PHGarfield velocity uses z_pad/time; match data's pad-to-CM distance.
            model_velocity = float(ref.velocity_cm_us) * distance / float(ref.z_pad_cm)
            row.update(
                phgarfield_velocity_cm_us=model_velocity,
                phgarfield_reported_velocity_cm_us=float(ref.velocity_cm_us),
                phgarfield_t0_bins=float(ref.t0_bins),
                reference_source=ref.source,
            )
        if info["status"] == "complete":
            with np.load(DATA_DIR / record["file"], allow_pickle=False) as source:
                edges = source[info["key"] + "_xedges"]
                counts = source[info["key"] + "_counts"]
            row.update(
                drift_metrics(
                    edges,
                    counts,
                    info["source"],
                    RAW_TIME_SHIFT_BINS,
                    TIME_BIN_NS,
                    distance,
                    T0_WINDOWS,
                    CM_WINDOWS,
                ),
                timing_source=info["source"],
            )
            if row["status"] == "estimated" and len(reference) == 1:
                row["delta_t0_bins"] = row["t0_bins"] - row["phgarfield_t0_bins"]
                row["velocity_ratio"] = row["velocity_cm_us"] / row["phgarfield_velocity_cm_us"]
            if run == FOCUS_RUN and group == GROUP:
                x = (edges[:-1] + edges[1:]) / 2
                fig, axes = plt.subplots(1, 3, figsize=(16, 4))
                shift = row["raw_shift_bins"]
                axes[0].plot(x, counts)
                axes[0].set(xlabel="stored time bin", title=info["side"] + " full timing spectrum")
                for ax, windows, title in zip(axes[1:], (T0_WINDOWS, CM_WINDOWS), ("t0", "CM")):
                    ax.plot(x - shift, counts)
                    ax.set_xlim(windows[0][0], windows[1][1])
                    ax.set(xlabel="unpacked time bin", title=title + "; " + row["status"])
                    value = row["t0_bins"] if title == "t0" else row["cm_bins"]
                    if np.isfinite(value):
                        ax.axvline(value, color="tab:red")
                fig.tight_layout()
                plt.show()
                plt.close(fig)
        else:
            row["status"] = "timing_unavailable"
        timing_rows.append(row)
TIMING_SUMMARY = pd.DataFrame(timing_rows)
# Ensure numeric columns exist even when no production timing histograms were stored.
for name in (
    "t0_bins",
    "cm_bins",
    "velocity_cm_us",
    "delta_t0_bins",
    "velocity_ratio",
    "phgarfield_velocity_cm_us",
    "phgarfield_t0_bins",
):
    if name not in TIMING_SUMMARY:
        TIMING_SUMMARY[name] = np.nan
print("Timing method: smoothed sideband 50% midpoint. Edge widths are not measurement errors.")
display(TIMING_SUMMARY)
for group, data in TIMING_SUMMARY.groupby("group"):
    runs = sorted(data.run.unique())
    position = {r: i for i, r in enumerate(runs)}
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    for side, color in [("south", "tab:blue"), ("north", "tab:red")]:
        rows = data[data.side == side].sort_values("run")
        x = [position[r] for r in rows.run]
        axes[0, 0].plot(x, rows.t0_bins, "o-", ms=3, color=color, label=side + " data")
        axes[0, 0].plot(
            x, rows.phgarfield_t0_bins, "--", color=color, label=side + " PHGarfield setting"
        )
        axes[0, 1].plot(x, rows.velocity_cm_us, "o-", ms=3, color=color, label=side + " data")
        axes[0, 1].plot(
            x, rows.phgarfield_velocity_cm_us, "--", color=color, label=side + " PHGarfield"
        )
        axes[1, 0].plot(x, rows.delta_t0_bins, "o-", ms=3, color=color, label=side)
        axes[1, 1].plot(x, rows.velocity_ratio, "o-", ms=3, color=color, label=side)
    for ax, label in zip(
        axes.flat,
        (
            "t0 [unpacked bins]",
            "drift velocity [cm/us]",
            "data t0 - PHGarfield setting [bins]",
            "data / PHGarfield velocity",
        ),
    ):
        ax.set_ylabel(label)
        ax.set_xticks(range(len(runs)))
        ax.set_xticklabels(runs, rotation=90)
        ax.legend(fontsize=8)
    axes[1, 0].axhline(0, color="gray", ls="--")
    axes[1, 1].axhline(1, color="gray", ls="--")
    fig.suptitle(group + " timing; current-CDB PHGarfield baseline, not reconstruction provenance")
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## Run trends and small CSV outputs
Every run has one row per side and pT window, including unavailable estimates.
Orange open symbols mark estimates using high-deviance slices. Reference-difference
errors below use an independent-run approximation; points share the reference error.
The large input maps are not copied into these small CSV files.


In [ ]:
if PILOT_APPROVED and RUN_BATCH and not RUN_SUMMARY.empty:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    RUN_SUMMARY.to_csv(OUTPUT_DIR / "rdca_odd_robust_per_run.csv", index=False)
    TIMING_SUMMARY.to_csv(OUTPUT_DIR / "timing_per_run.csv", index=False)
    if not CONDITIONS.empty:
        CONDITIONS.to_csv(OUTPUT_DIR / "conditions_snapshot.csv", index=False)
    if not PHGARFIELD.empty:
        PHGARFIELD.to_csv(OUTPUT_DIR / "phgarfield_reference.csv", index=False)
    PROFILE_SUMMARY.to_csv(OUTPUT_DIR / "slice_fit_quality.csv", index=False)
    CHARGE_SUMMARY.to_csv(OUTPUT_DIR / "charge_balance.csv", index=False)
    configuration = dict(
        data=str(DATA_DIR.resolve()),
        fit_settings=FIT_SETTINGS,
        odd_definition="(mu_positive - mu_negative)/2 [cm]",
        full_difference_definition="mu_positive - mu_negative",
        estimator="Huber c + a sin(phi) + b cos(phi)",
        bootstrap="resample paired phi bins; not a segment bootstrap",
        bootstrap_repeats=BOOTSTRAP_REPEATS,
        references=REFERENCE_RUNS,
        calibration_applied=False,
        raw_time_shift_bins=RAW_TIME_SHIFT_BINS,
        time_bin_ns=TIME_BIN_NS,
        fallback_distance_cm=DRIFT_DISTANCE_CM,
        t0_windows=T0_WINDOWS,
        cm_windows=CM_WINDOWS,
        conditions=condition_info,
    )
    (OUTPUT_DIR / "settings.json").write_text(json.dumps(configuration, indent=2))
    for (group, window), data in RUN_SUMMARY.groupby(["group", "window"]):
        runs = sorted(data.run.unique())
        positions = {r: i for i, r in enumerate(runs)}
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        for side, color in [("south", "tab:blue"), ("north", "tab:red")]:
            rows = data[data.side == side].sort_values("run")
            x = np.array([positions[r] for r in rows.run])
            good = rows.status == "estimated"
            axes[0].errorbar(
                x[good],
                rows.loc[good, "odd_half_mm"],
                yerr=10 * rows.loc[good, "approximate_error"],
                fmt="o-",
                ms=4,
                color=color,
                label=side,
            )
            flagged = good & rows.model_mismatch_flag
            axes[0].plot(
                x[flagged],
                rows.loc[flagged, "odd_half_mm"],
                "o",
                mfc="none",
                mec="orange",
                ms=9,
                label=side + " high-deviance inputs" if flagged.any() else None,
            )
            axes[1].errorbar(
                x,
                rows.delta_common_odd_half_mm,
                yerr=rows.delta_common_approximate_error_mm,
                fmt="o-",
                ms=4,
                color=color,
                label=side,
            )
        axes[0].set_title("Robust odd constant; per-run valid phi bins")
        axes[1].set_title("Common-mask odd change from reference")
        step = max(1, int(np.ceil(len(runs) / 12)))
        for ax in axes:
            ax.axhline(0, color="gray", ls="--", lw=1)
            ax.set_xticks(
                range(0, len(runs), step), [str(r) for r in runs[::step]], rotation=60, ha="right"
            )
            ax.set(xlabel="Run", ylabel="Half-difference [mm]")
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                ax.legend(fontsize=8)
        fig.suptitle(
            group + "; " + window + "; working threshold " + str(FIT_SETTINGS["max_deviance_ndf"])
        )
        fig.tight_layout()
        if SAVE_FIGURES:
            fig.savefig(OUTPUT_DIR / (group.replace("/", "_") + "_" + window + "_odd.png"), dpi=150)
        plt.show()
        plt.close(fig)
    print("Saved:", OUTPUT_DIR.resolve())
    print("Main table: rdca_odd_robust_per_run.csv")

In [ ]:
print("Dataset:", DATA_DIR.resolve())
print("Conditions:", MANIFEST.get("conditions", "not exported"))
print("First run:", MANIFEST["runs"][0]["run"])
print("Timing:", MANIFEST["runs"][0].get("timing", "not exported"))